LLM-based Feature Engineering using Ollama (Fully Free)

This notebook builds a fraud detection pipeline using:
- Ollama (local LLM)
- Behavioral feature generation
- DistilBERT model
- Iterative loop (semi-automated)


STEP 0: Install Ollama (RUN IN TERMINAL, NOT NOTEBOOK)

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3
ollama run llama3
```

Keep Ollama running in background.

In [1]:
# STEP 1: Install dependencies
!pip install pandas numpy scikit-learn transformers datasets tqdm requests

In [2]:
# STEP 2: Load dataset
import pandas as pd

df = pd.read_csv('../../transactions/card_transaction.v1.csv')
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [4]:
# STEP 3: Create user-level fraud label

# ---- Clean Amount ----
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)

# ---- Clean Use Chip ----
df['Use Chip'] = df['Use Chip'].map({'Yes': 1, 'No': 0}).fillna(0)

# ---- Clean Fraud column ----
df['Is Fraud?'] = (
    df['Is Fraud?']
    .astype(str)
    .str.strip()
    .map({'Yes': 1, 'No': 0, '1': 1, '0': 0})
)

# Handle any unexpected values
df['Is Fraud?'] = df['Is Fraud?'].fillna(0).astype(int)

# ---- Create User-level label ----
fraud_users = df.groupby('User')['Is Fraud?'].max().reset_index()
fraud_users.columns = ['User', 'User_Fraud_Label']

df = df.merge(fraud_users, on='User')
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,User_Fraud_Label
0,0,0,2002,9,1,06:21,134.09,0.0,3527213246127876953,La Verne,CA,91750.0,5300,NaN,0,1
1,0,0,2002,9,1,06:42,38.48,0.0,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,1
2,0,0,2002,9,2,06:22,120.34,0.0,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,1
3,0,0,2002,9,2,17:45,128.95,0.0,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,0,1
4,0,0,2002,9,3,06:23,104.71,0.0,5817218446178736267,La Verne,CA,91750.0,5912,NaN,0,1


In [5]:
# STEP 4: Connect to Ollama
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()
    
    if "response" in data:
        return data["response"]
    else:
        print("Error:", data)
        return None

In [6]:
# STEP 5: Generate features using LLM

schema = list(df.columns)

prompt = f"""
Dataset columns: {schema}

Generate fraud detection features.


IMPORTANT:
- Output VALID pandas code using df
- Use groupby('User') when needed
- Ensure output aligns with dataframe length
- Convert categorical variables before aggregation

Example:
df['avg_amount'] = df.groupby('User')['Amount'].transform('mean')

STRICT JSON ONLY:
[
  {{
    "feature_name": "",
    "description": "",
    "type": "numerical/categorical",
    "computation": "VALID PYTHON CODE",
    "intuition": ""
  }}
]
"""

llm_output = call_llm(prompt)
print(llm_output)

Here are the fraud detection features generated:

```
[
  {
    "feature_name": "avg_amount_per_user",
    "description": "Average transaction amount per user",
    "type": "numerical",
    "computation": "df['avg_amount_per_user'] = df.groupby('User')['Amount'].transform('mean')",
    "intuition": "Users with higher average transaction amounts may be more likely to be fraudulent"
  },
  {
    "feature_name": "num_transactions_per_year_per_user",
    "description": "Number of transactions per year per user",
    "type": "numerical",
    "computation": "df['num_transactions_per_year_per_user'] = df.groupby(['User', 'Year'])['Amount'].count().reset_index(name='num_transactions').groupby('User')['num_transactions'].transform('sum')",
    "intuition": "Users with a high number of transactions per year may be more likely to be fraudulent"
  },
  {
    "feature_name": "num_fraudulent_transactions_per_user",
    "description": "Number of fraudulent transactions per user",
    "type": "numeric

In [7]:
def extract_json(llm_output):
    try:
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []

In [8]:
def apply_features_exec(df, features):
    for feature in features:
        code = feature.get("computation", "")
        print("\nExecuting:\n", code)

        try:
            exec(code)
        except Exception as e:
            print("❌ Error:", e)

    return df


features = extract_json(llm_output)
df = apply_features_exec(df, features)

df.head()

Failed to parse JSON
Here are the fraud detection features generated:

```
[
  {
    "feature_name": "avg_amount_per_user",
    "description": "Average transaction amount per user",
    "type": "numerical",
    "computation": "df['avg_amount_per_user'] = df.groupby('User')['Amount'].transform('mean')",
    "intuition": "Users with higher average transaction amounts may be more likely to be fraudulent"
  },
  {
    "feature_name": "num_transactions_per_year_per_user",
    "description": "Number of transactions per year per user",
    "type": "numerical",
    "computation": "df['num_transactions_per_year_per_user'] = df.groupby(['User', 'Year'])['Amount'].count().reset_index(name='num_transactions').groupby('User')['num_transactions'].transform('sum')",
    "intuition": "Users with a high number of transactions per year may be more likely to be fraudulent"
  },
  {
    "feature_name": "num_fraudulent_transactions_per_user",
    "description": "Number of fraudulent transactions per user",

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,User_Fraud_Label
0,0,0,2002,9,1,06:21,134.09,0.0,3527213246127876953,La Verne,CA,91750.0,5300,NaN,0,1
1,0,0,2002,9,1,06:42,38.48,0.0,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,1
2,0,0,2002,9,2,06:22,120.34,0.0,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,1
3,0,0,2002,9,2,17:45,128.95,0.0,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,0,1
4,0,0,2002,9,3,06:23,104.71,0.0,5817218446178736267,La Verne,CA,91750.0,5912,NaN,0,1


In [29]:
# # Select feature columns properly
# exclude_cols = ['User', 'User_Fraud_Label']
# #exclude_cols = ['User']

# feature_cols = [
#     col for col in df.select_dtypes(include=['int64', 'float64']).columns
#     if col not in exclude_cols
# ]

# # Aggregate
# user_df = df.groupby('User')[feature_cols].mean().reset_index()

# # Add label
# labels = df.groupby('User')['User_Fraud_Label'].max().reset_index()

# user_df = user_df.merge(labels, on='User')

# user_df.head()


# -------- STEP 1: Feature selection --------
exclude_cols = ['User', 'User_Fraud_Label', 'Is Fraud?']

feature_cols = [
    col for col in df.select_dtypes(include=['int64', 'float64']).columns
    if col not in exclude_cols
]

# -------- STEP 2: Aggregate features --------
user_df = df.groupby('User')[feature_cols].mean().reset_index()

# -------- STEP 3: Add label --------
labels = df[['User', 'User_Fraud_Label']].drop_duplicates()

user_df = user_df.merge(labels, on='User')

# -------- STEP 4: Prepare X and y --------
X = user_df.drop(columns=['User', 'User_Fraud_Label'])
y = user_df['User_Fraud_Label']

# -------- STEP 5: Check class distribution --------
print("Total samples:", len(y))
print("Class distribution:\n", y.value_counts())



Total samples: 2000
Class distribution:
 User_Fraud_Label
1    1343
0     657
Name: count, dtype: int64


In [30]:
from sklearn.model_selection import train_test_split

X = user_df.drop(columns=['User', 'User_Fraud_Label'])
y = user_df['User_Fraud_Label']

print("Total samples:", len(y))
print("Total samples:", len(y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Total samples: 2000
Total samples: 2000


In [31]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=42,
    #n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

,n_estimators,50
,criterion,'gini'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.865
Precision: 0.8745644599303136
Recall: 0.9330855018587361
F1 Score: 0.9028776978417267
ROC-AUC: 0.9246289622293481


In [33]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[ 95  36]
 [ 18 251]]


In [34]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print(importances.head(10))

Year             0.415215
Month            0.255030
Card             0.128181
Day              0.090816
MCC              0.031882
Amount           0.028115
Merchant Name    0.025537
Zip              0.025224
Use Chip         0.000000
dtype: float64


In [35]:
# # STEP 7: Convert features to text for DistilBERT

# def row_to_text(row):
#     return f"User behavior: avg {row['avg_amount']}, count {row['txn_count']}, std {row['amount_std']}"

# df['text'] = df.apply(row_to_text, axis=1)
# df[['text', 'User_Fraud_Label']].head()

In [36]:
# # STEP 8: Train DistilBERT
# from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
# from datasets import Dataset

# dataset = Dataset.from_pandas(df[['text', 'User_Fraud_Label']])

# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# def tokenize(example):
#     return tokenizer(example['text'], truncation=True, padding='max_length')

# dataset = dataset.map(tokenize, batched=True)
# dataset = dataset.train_test_split(test_size=0.2)

# model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=2,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     evaluation_strategy='epoch'
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset['train'],
#     eval_dataset=dataset['test']
# )

# trainer.train()

In [37]:
# # STEP 9: Evaluate
# metrics = trainer.evaluate()
# print(metrics)

 STEP 10: Iteration (Manual + Guided)

If performance is low:
- Re-run STEP 5 with improved prompt
- Add/remove features
- Retrain model

Optional: Add feedback into prompt like:

Model struggling with temporal patterns → generate time-based features
